# Round 3 Manual Challenge: Bio-Pods Two-Bid Optimization

This notebook solves the manual challenge as a parameterized game-theory/optimization problem.

## Goal
Choose two bids (`b1`, `b2`) in `[670, 920]` with step `5` to maximize expected profit per counterparty.

## Mechanics used (from `wiki/round3manual.md`)
- Reserve price `R` is uniformly distributed over `{670, 675, ..., 920}`.
- If `b1 > R`, you trade at `b1`.
- Else, if `b2 > R`:
  - If `b2 > avg_b2`, second-bid trade is treated as full value.
  - If `b2 <= avg_b2`, second-bid outcome is penalized by
  
  \[
  \left(\frac{920-avg\_b2}{920-b2}\right)^3
  \]
- Next day sale value is fixed at `920`.

## Important modeling assumption
The text says second-bid trade probability drops when `b2 <= avg_b2` and also gives a PnL penalty factor.  
Here, that factor is implemented directly as an expected-value multiplier on second-bid PnL when `b2 <= avg_b2`.

This is a conservative and internally consistent approximation.

In [1]:
import numpy as np
import pandas as pd

RESALE_PRICE = 920
BID_MIN, BID_MAX, STEP = 670, 920, 5

RESERVES = np.arange(BID_MIN, BID_MAX + STEP, STEP)  # uniform discrete
BIDS = np.arange(BID_MIN, BID_MAX + STEP, STEP)
AVG_GRID = np.arange(760, 901, STEP)  # plausible range for average second bid

print(f"Reserve support size: {len(RESERVES)}")
print(f"Bid support size: {len(BIDS)}")

Reserve support size: 51
Bid support size: 51


In [2]:
def second_bid_multiplier(b2: int, avg_b2: float, resale_price: int = RESALE_PRICE) -> float:
    """Penalty multiplier when b2 <= avg_b2."""
    if b2 > avg_b2:
        return 1.0
    # Guard against division edge (b2=920).
    if b2 >= resale_price:
        return 0.0
    ratio = (resale_price - avg_b2) / (resale_price - b2)
    return float(np.clip(ratio ** 3, 0.0, 1.0))


def expected_profit_per_counterparty(b1: int, b2: int, avg_b2: float) -> float:
    """
    EV under a fixed avg second bid and uniform reserve prices.

    First bid gets first priority.
    Second bid only applies to counterparties not filled by first bid.
    """
    if b2 < b1:
        return -1e18

    first_hit = (RESERVES < b1)
    second_hit = (RESERVES >= b1) & (RESERVES < b2)

    p1 = first_hit.mean()
    p2 = second_hit.mean()

    m2 = second_bid_multiplier(b2, avg_b2)

    ev1 = (RESALE_PRICE - b1) * p1
    ev2 = (RESALE_PRICE - b2) * p2 * m2
    return float(ev1 + ev2)


def optimize_for_fixed_avg(avg_b2: float) -> pd.Series:
    rows = []
    for b1 in BIDS:
        for b2 in BIDS:
            if b2 < b1:
                continue
            ev = expected_profit_per_counterparty(int(b1), int(b2), float(avg_b2))
            rows.append((ev, int(b1), int(b2)))
    best = max(rows, key=lambda x: x[0])
    return pd.Series({"avg_b2": avg_b2, "ev": best[0], "b1": best[1], "b2": best[2]})

## 1) Optimal bids if `avg_b2` were known exactly

This gives a strategy map by scenario. In practice, you do not know `avg_b2`, but this reveals where the objective changes regime.

In [3]:
fixed_avg_opt = pd.DataFrame([optimize_for_fixed_avg(a) for a in AVG_GRID])
fixed_avg_opt.head(12)

,avg_b2,ev,b1,b2
0,760.0,81.666667,750.0,835.0
1,765.0,81.666667,750.0,835.0
2,770.0,81.666667,750.0,835.0
3,775.0,81.666667,750.0,835.0
4,780.0,81.666667,750.0,835.0
5,785.0,81.666667,750.0,835.0
6,790.0,81.666667,750.0,835.0
7,795.0,81.666667,750.0,835.0
8,800.0,81.666667,750.0,835.0
9,805.0,81.666667,750.0,835.0


## 2) Robust maximin choice (uncertain `avg_b2`)

If you want a conservative choice that avoids downside across many possible opponent averages, maximize the **worst-case EV** over the scenario grid.

In [4]:
robust_rows = []
for b1 in BIDS:
    for b2 in BIDS:
        if b2 < b1:
            continue
        vals = np.array([expected_profit_per_counterparty(int(b1), int(b2), float(a)) for a in AVG_GRID])
        robust_rows.append({
            "b1": int(b1),
            "b2": int(b2),
            "ev_min": float(vals.min()),
            "ev_mean": float(vals.mean()),
            "ev_max": float(vals.max()),
        })

robust_df = pd.DataFrame(robust_rows).sort_values(["ev_min", "ev_mean"], ascending=False)
robust_df.head(10)

,b1,b2,ev_min,ev_mean,ev_max
943,785,900,69.901961,69.901961,69.901961
915,780,900,69.803922,69.803922,69.803922
970,790,900,69.803922,69.803922,69.803922
886,775,900,69.509804,69.509804,69.509804
996,795,900,69.509804,69.509804,69.509804
856,770,900,69.019608,69.019608,69.019608
1021,800,900,69.019608,69.019608,69.019608
825,765,900,68.333333,68.333333,68.333333
1045,805,900,68.333333,68.333333,68.333333
944,785,905,67.941176,67.941176,67.941176


## 3) Bayesian choice (if you have a belief about `avg_b2`)

Assume `avg_b2` is random with a prior centered around likely opponent behavior.  
Default below: Normal-like weights on grid with mean `850`, std `25`.

Change `MU` and `SIGMA` to match your belief.

In [5]:
MU, SIGMA = 850, 25
weights = np.exp(-0.5 * ((AVG_GRID - MU) / SIGMA) ** 2)
weights = weights / weights.sum()

bayes_rows = []
for b1 in BIDS:
    for b2 in BIDS:
        if b2 < b1:
            continue
        vals = np.array([expected_profit_per_counterparty(int(b1), int(b2), float(a)) for a in AVG_GRID])
        bayes_rows.append({
            "b1": int(b1),
            "b2": int(b2),
            "ev_bayes": float((vals * weights).sum()),
            "ev_min": float(vals.min()),
            "ev_max": float(vals.max()),
        })

bayes_df = pd.DataFrame(bayes_rows).sort_values(["ev_bayes"], ascending=False)
bayes_df.head(10)

,b1,b2,ev_bayes,ev_min,ev_max
849,770,865,76.693623,59.808783,79.313725
879,775,865,76.635445,60.639281,79.117647
850,770,870,76.573621,60.078431,78.431373
880,775,870,76.568470,60.898039,78.333333
818,765,865,76.555723,58.782207,79.313725
848,770,860,76.475124,59.607843,80.000000
819,765,870,76.382695,59.062745,78.333333
908,780,865,76.381188,61.273700,78.725490
817,765,860,76.377336,58.572985,80.098039
878,775,860,76.376833,60.446623,79.705882


In [6]:
print("Robust (maximin) top choice:")
print(robust_df.iloc[0].to_dict())

print("\nBayesian top choice (MU=850, SIGMA=25):")
print(bayes_df.iloc[0].to_dict())

# Useful candidate shortlist for submission
candidate_pairs = [
    (785, 900),  # robust maximin from default grid
    (775, 865),  # high expected value under MU~850 prior
    (770, 870),  # near-optimal Bayesian alternative
]

print("\nCandidate EV table by avg_b2 scenario:")
rows = []
for b1, b2 in candidate_pairs:
    vals = np.array([expected_profit_per_counterparty(b1, b2, float(a)) for a in AVG_GRID])
    rows.append({
        "b1": b1,
        "b2": b2,
        "ev_min": float(vals.min()),
        "ev_mean": float(vals.mean()),
        "ev_at_850": float(expected_profit_per_counterparty(b1, b2, 850)),
        "ev_max": float(vals.max()),
    })

pd.DataFrame(rows).sort_values("ev_mean", ascending=False)

Robust (maximin) top choice:
{'b1': 785.0, 'b2': 900.0, 'ev_min': 69.90196078431373, 'ev_mean': 69.90196078431374, 'ev_max': 69.90196078431373}

Bayesian top choice (MU=850, SIGMA=25):
{'b1': 770.0, 'b2': 865.0, 'ev_bayes': 76.69362329300053, 'ev_min': 59.80878301733917, 'ev_max': 79.31372549019608}

Candidate EV table by avg_b2 scenario:


,b1,b2,ev_min,ev_mean,ev_at_850,ev_max
1,775,865,60.639281,75.935242,79.117647,79.117647
2,770,870,60.078431,75.719405,78.431373,78.431373
0,785,900,69.901961,69.901961,69.901961,69.901961


## Practical recommendation from current assumptions

- **Conservative / robust pick:** `b1=785`, `b2=900`
  - Best worst-case EV over a broad `avg_b2` range (`760..900`).
  - Intuition: very high `b2` avoids severe below-average penalty scenarios.

- **More aggressive expectation-based pick:** `b1=775`, `b2=865`
  - Higher EV if you believe field average second bid is around `840-870`.
  - More exposed if true `avg_b2` is significantly higher.

If unsure about others' behavior, the robust pair is usually safer for leaderboard regret minimization.

## Last-step checklist before submitting
1. Re-run this notebook with your latest belief for `MU`, `SIGMA`, and scenario grid.
2. Confirm candidate ranking remains stable.
3. Submit the selected pair in the manual challenge UI.

---

### Notes for extension
- You can add a game-theoretic fixed-point loop where `avg_b2` is induced by all players running the same optimization under heterogeneous beliefs.
- You can also model a separate explicit probability-of-fill function for `b2 <= avg_b2` if new information appears.

## 4) Mean-field fixed-point model for `avg_b2`

This section models strategic interaction directly.

### Idea
- Suppose each participant best-responds to a conjectured crowd average second bid `A`.
- Given `A`, each participant type chooses `(b1, b2)` maximizing its own objective.
- Those choices imply a new aggregate average second bid `A'`.
- A fixed point satisfies `A' = A`.

We solve this by iterative best response with type heterogeneity.

In [7]:
TYPE_CONFIG = [
    # weight, belief_sigma, risk_penalty_lambda
    {"name": "rational_tight", "weight": 0.45, "sigma": 8,  "risk_lambda": 0.05},
    {"name": "rational_wide",  "weight": 0.30, "sigma": 18, "risk_lambda": 0.10},
    {"name": "cautious",       "weight": 0.20, "sigma": 25, "risk_lambda": 0.25},
    {"name": "chaotic",        "weight": 0.05, "sigma": 35, "risk_lambda": 0.00},
]


def scenario_grid(center, sigma, width_mult=3, step=STEP):
    lo = max(BID_MIN, int(np.floor((center - width_mult * sigma) / step) * step))
    hi = min(BID_MAX, int(np.ceil((center + width_mult * sigma) / step) * step))
    return np.arange(lo, hi + step, step)


def utility_under_conjecture(b1, b2, center_avg, sigma, risk_lambda=0.0):
    avgs = scenario_grid(center_avg, sigma)
    w = np.exp(-0.5 * ((avgs - center_avg) / max(sigma, 1e-6)) ** 2)
    w = w / w.sum()
    vals = np.array([expected_profit_per_counterparty(b1, b2, float(a)) for a in avgs])
    ev = float((vals * w).sum())
    sd = float(np.sqrt((w * (vals - ev) ** 2).sum()))
    return ev - risk_lambda * sd


def best_response_pair(center_avg, sigma, risk_lambda):
    best_u = -1e18
    best_pair = None
    for b1 in BIDS:
        for b2 in BIDS:
            if b2 < b1:
                continue
            u = utility_under_conjecture(int(b1), int(b2), center_avg, sigma, risk_lambda)
            if u > best_u:
                best_u = u
                best_pair = (int(b1), int(b2), float(u))
    return best_pair

In [8]:
def iterate_mean_field_fixed_point(init_avg=850, iters=25, damping=0.6):
    A = float(init_avg)
    history = []

    for t in range(iters):
        type_rows = []
        implied_b2 = 0.0

        for cfg in TYPE_CONFIG:
            b1, b2, u = best_response_pair(A, cfg["sigma"], cfg["risk_lambda"])
            wt = cfg["weight"]
            implied_b2 += wt * b2
            type_rows.append({
                "type": cfg["name"],
                "weight": wt,
                "sigma": cfg["sigma"],
                "risk_lambda": cfg["risk_lambda"],
                "br_b1": b1,
                "br_b2": b2,
                "utility": u,
            })

        A_new = damping * implied_b2 + (1 - damping) * A
        history.append({
            "iter": t,
            "A_prev": A,
            "A_implied": implied_b2,
            "A_next": A_new,
            "gap": implied_b2 - A,
            "type_rows": type_rows,
        })

        A = A_new
        if abs(history[-1]["gap"]) < 1e-3:
            break

    return A, history


A_star, fp_history = iterate_mean_field_fixed_point(init_avg=850, iters=30, damping=0.65)
fp_summary = pd.DataFrame([
    {
        "iter": h["iter"],
        "A_prev": round(h["A_prev"], 3),
        "A_implied": round(h["A_implied"], 3),
        "A_next": round(h["A_next"], 3),
        "gap": round(h["gap"], 3),
    }
    for h in fp_history
])

print(f"Estimated fixed point avg_b2*: {A_star:.3f}")
fp_summary.tail(10)

Estimated fixed point avg_b2*: 889.500


,iter,A_prev,A_implied,A_next,gap
7,7,884.020,887.0,885.957,2.980
8,8,885.957,889.5,888.260,3.543
9,9,888.260,889.5,889.066,1.240
10,10,889.066,889.5,889.348,0.434
11,11,889.348,889.5,889.447,0.152
12,12,889.447,889.5,889.481,0.053
13,13,889.481,889.5,889.493,0.019
14,14,889.493,889.5,889.498,0.007
15,15,889.498,889.5,889.499,0.002
16,16,889.499,889.5,889.500,0.001


In [9]:
last_types = pd.DataFrame(fp_history[-1]["type_rows"]).sort_values("weight", ascending=False)
last_types

,type,weight,sigma,risk_lambda,br_b1,br_b2,utility
0,rational_tight,0.45,8,0.05,785,890,70.569815
1,rational_wide,0.30,18,0.10,785,890,68.913928
2,cautious,0.20,25,0.25,785,890,68.136042
3,chaotic,0.05,35,0.00,785,880,70.004346


In [10]:
# "Our" best response to the estimated equilibrium average

eq_avg = float(round(A_star / STEP) * STEP)
our_best = optimize_for_fixed_avg(eq_avg)
our_best

avg_b2    890.000000
ev         73.333333
b1        780.000000
b2        890.000000
dtype: float64

### Interpreting the fixed point

- `A_star` is the self-consistent crowd average second bid under this model.
- `last_types` shows each type's best-response bid pair at convergence.
- `our_best` gives the best response if we believe the crowd settles near this equilibrium.

You should stress-test by changing `TYPE_CONFIG` (weights, sigmas, risk aversion), then rerunning this section.

If `A_star` is stable across reasonable perturbations, confidence in the final bid pair is higher.

## 5) Past-winner manual playbook (transferable patterns)

Synthesized from top-team writeups/references (Frankfurt Hedgehogs, CMU Physics, Alpha Animals notes, and Prosperity 4 Fisjo repo).

### What repeatedly worked
1. **Reduce to explicit EV formulas first.**
   - Winners wrote down payoff equations and solved analytically when possible.
2. **Use game-theory framing for crowd-dependent terms.**
   - Nash / best-response logic for average-dependent payoffs.
3. **Model opponent heterogeneity, not just pure rationality.**
   - Add priors for Nash players, random players, and "psychological" clusters.
4. **Use minimax when qualification thresholds matter.**
   - Some teams deliberately chose guaranteed lower variance outcomes over higher-EV gambles.
5. **Run sensitivity, not single-point optimization.**
   - Evaluate stability over plausible crowd-parameter ranges.

### Round-3 reserve-price style precedent (Prosperity 3)
- Teams commonly solved a one-bid benchmark first, then extended to two-bid + average/penalty interactions.
- Strong teams explicitly recognized asymmetry: being below average can hurt more than being above it helps.
- Practical response was often to bias second bid slightly above naive Nash to avoid penalty cliffs.

### Implication for this Round 3 Bio-Pods challenge
- Do not rely on a single guessed `avg_b2`.
- Use all three views together:
  - scenario-optimal,
  - robust maximin,
  - mean-field fixed-point.
- Prefer stable bid pairs that remain near-optimal across these views.

### Sources used in this synthesis
- Local references in this repo:
  - `reference/timodiehm-p3/README.md`
  - `reference/chrispyroberts-p3/readme.md`
  - `reference/alphaanimals-p3/README.md`
  - `reference/fisjo-p4/README.md`
  - `reference/chrispyroberts-p3/ROUND 3/manualn.ipynb`
- External public writeups:
  - [TimoDiehm/imc-prosperity-3](https://github.com/TimoDiehm/imc-prosperity-3)
  - [chrispyroberts/imc-prosperity-3](https://github.com/chrispyroberts/imc-prosperity-3)
  - [angus4718/imc-prosperity-3-public](https://github.com/angus4718/imc-prosperity-3-public)
  - [gabsens/IMC-Prosperity-2-Manual](https://github.com/gabsens/IMC-Prosperity-2-Manual)

## 6) Simple final submission recommender

This intentionally avoids overcomplication.

We score each candidate pair by a weighted blend of:
- **Robustness** (worst-case EV over scenario grid)
- **Bayesian EV** (expected EV under your prior)
- **Fixed-point consistency** (EV at rounded equilibrium `avg_b2*`)

Default weights (easy to interpret):
- robust: `0.40`
- bayes: `0.35`
- fixed-point: `0.25`

You can change the weights, but keep them summing to 1.

In [11]:
# Blend weights (must sum to 1)
W_ROBUST = 0.40
W_BAYES = 0.35
W_FIXED = 0.25

w_sum = W_ROBUST + W_BAYES + W_FIXED
assert abs(w_sum - 1.0) < 1e-9, "Weights must sum to 1."

A_eq = float(round(A_star / STEP) * STEP)

# Build a unified candidate table
cand = robust_df[["b1", "b2", "ev_min", "ev_mean", "ev_max"]].copy()
cand = cand.merge(bayes_df[["b1", "b2", "ev_bayes"]], on=["b1", "b2"], how="left")
cand["ev_fixed"] = cand.apply(
    lambda r: expected_profit_per_counterparty(int(r.b1), int(r.b2), A_eq), axis=1
)

# Normalize each component to [0,1] to combine fairly
for col in ["ev_min", "ev_bayes", "ev_fixed"]:
    cmin, cmax = cand[col].min(), cand[col].max()
    if cmax - cmin < 1e-12:
        cand[col + "_norm"] = 0.0
    else:
        cand[col + "_norm"] = (cand[col] - cmin) / (cmax - cmin)

cand["final_score"] = (
    W_ROBUST * cand["ev_min_norm"]
    + W_BAYES * cand["ev_bayes_norm"]
    + W_FIXED * cand["ev_fixed_norm"]
)

recommendations = cand.sort_values(["final_score", "ev_min", "ev_bayes"], ascending=False)

topn = recommendations.head(10).copy()

print(f"Rounded fixed-point avg_b2 used in recommender: {A_eq:.0f}")
print("\nTop recommendation:")
print(topn.iloc[0][["b1", "b2", "final_score", "ev_min", "ev_bayes", "ev_fixed"]].to_dict())

print("\nTop 10 candidates:")
topn[["b1", "b2", "final_score", "ev_min", "ev_bayes", "ev_fixed"]]

Rounded fixed-point avg_b2 used in recommender: 890

Top recommendation:
{'b1': 785.0, 'b2': 900.0, 'final_score': 0.957307621162603, 'ev_min': 69.90196078431373, 'ev_bayes': 69.90196078431372, 'ev_fixed': 69.90196078431373}

Top 10 candidates:


,b1,b2,final_score,ev_min,ev_bayes,ev_fixed
0,785,900,0.957308,69.901961,69.901961,69.901961
1,780,900,0.955965,69.803922,69.803922,69.803922
2,790,900,0.955965,69.803922,69.803922,69.803922
45,785,890,0.952397,64.542484,73.056059,73.235294
3,775,900,0.951937,69.509804,69.509804,69.509804
4,795,900,0.951937,69.509804,69.509804,69.509804
50,780,890,0.951332,64.226580,73.145563,73.333333
20,785,895,0.951096,66.403922,71.608808,71.666667
42,790,890,0.950777,64.662309,72.770476,72.941176
19,790,895,0.949792,66.447059,71.415359,71.470588


### Practical use

- If top 3 are tightly clustered, choose the one with highest `ev_min` (safer).
- If one candidate clearly dominates in both `ev_min` and `ev_bayes`, use it.
- Re-run after changing priors (`MU`, `SIGMA`) or type mix (`TYPE_CONFIG`).

## 7) Final submit-now output (single pair)

This cell returns exactly one bid pair.

Rule (deterministic, conservative):
1. Start from top blended-score candidates.
2. Keep only candidates with `ev_min` within 0.5% of best `ev_min`.
3. From those, choose highest `final_score`.
4. Tie-break by higher `b2` (safer vs penalty cliff), then higher `b1`.

In [12]:
# Requires `recommendations` from Section 6

best_ev_min = recommendations["ev_min"].max()
threshold = 0.995 * best_ev_min  # within 0.5% of best worst-case EV

pool = recommendations[recommendations["ev_min"] >= threshold].copy()

submit_row = pool.sort_values(
    ["final_score", "b2", "b1"],
    ascending=[False, False, False]
).iloc[0]

submit_b1 = int(submit_row["b1"])
submit_b2 = int(submit_row["b2"])

print("SUBMIT_NOW")
print({"b1": submit_b1, "b2": submit_b2})
print("Diagnostics:")
print({
    "best_ev_min": float(best_ev_min),
    "threshold_ev_min": float(threshold),
    "chosen_final_score": float(submit_row["final_score"]),
    "chosen_ev_min": float(submit_row["ev_min"]),
    "chosen_ev_bayes": float(submit_row["ev_bayes"]),
    "chosen_ev_fixed": float(submit_row["ev_fixed"]),
})

SUBMIT_NOW
{'b1': 785, 'b2': 900}
Diagnostics:
{'best_ev_min': 69.90196078431373, 'threshold_ev_min': 69.55245098039217, 'chosen_final_score': 0.957307621162603, 'chosen_ev_min': 69.90196078431373, 'chosen_ev_bayes': 69.90196078431372, 'chosen_ev_fixed': 69.90196078431373}
